# Comparison Plots with Seaborn

When the goal is to compare trends, group means, or distributions across multiple categories, Seaborn provides several tools that go beyond simple scatter or categorical plots. This notebook focuses on line plots, point plots, grouped comparisons, and facet grids.

Topics covered:
- `sns.lineplot()` — trends and time series
- `sns.pointplot()` — categorical means with confidence intervals connected by lines
- Grouped `sns.barplot()` and `sns.boxplot()` using `hue`
- `FacetGrid` via `relplot()` and `catplot()` with `col` / `row`
- Practical comparisons across multiple grouping variables

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
tips = sns.load_dataset('tips')
flights = sns.load_dataset('flights')

print(f'tips: {tips.shape}')
print(f'flights: {flights.shape}')
flights.head()

## Line Plots with `sns.lineplot()`

`lineplot()` draws lines connecting data points, making it the natural choice for time series and trend data. When there are multiple observations per x-value, it plots the **mean** and a **95% confidence interval** by default.

In [ ]:
# Passenger counts over time
plt.figure(figsize=(10, 5))
sns.lineplot(data=flights, x='year', y='passengers')
plt.title('Average Monthly Passengers per Year (with 95% CI)')
plt.show()

In [ ]:
# Line plot with hue to compare months
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=flights, x='year', y='passengers',
    hue='month', palette='tab10',
    marker='o', markersize=5,
)
plt.title('Passenger Trends by Month')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Using style to differentiate line patterns
# Create a season column for demonstration
season_map = {
    'Jan': 'Winter', 'Feb': 'Winter', 'Mar': 'Spring',
    'Apr': 'Spring', 'May': 'Spring', 'Jun': 'Summer',
    'Jul': 'Summer', 'Aug': 'Summer', 'Sep': 'Fall',
    'Oct': 'Fall', 'Nov': 'Fall', 'Dec': 'Winter',
}
flights['season'] = flights['month'].map(season_map)

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=flights, x='year', y='passengers',
    hue='season', style='season',
    markers=True, dashes=True,
)
plt.title('Passengers by Season')
plt.show()

### Controlling the Confidence Band

Use `errorbar` to change the uncertainty representation:
- `errorbar='ci'` (default) — bootstrapped confidence interval
- `errorbar='sd'` — standard deviation
- `errorbar=None` — no band

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, eb, label in zip(
    axes,
    [('ci', 95), 'sd', None],
    ['95% CI', 'Std Dev', 'None'],
):
    sns.lineplot(
        data=flights, x='year', y='passengers',
        errorbar=eb, ax=ax,
    )
    ax.set_title(f'errorbar = {label}')

plt.tight_layout()
plt.show()

## Point Plots: `sns.pointplot()`

`pointplot()` is similar to `barplot()` but uses dots and lines rather than bars. The connecting lines make it easy to see trends across an ordered categorical axis. This is especially useful for interaction effects.

In [ ]:
plt.figure(figsize=(8, 5))
sns.pointplot(
    data=tips, x='day', y='total_bill',
    order=['Thur', 'Fri', 'Sat', 'Sun'],
)
plt.title('Mean Total Bill by Day')
plt.show()

In [ ]:
# Point plot with hue — spotting interaction effects
plt.figure(figsize=(8, 5))
sns.pointplot(
    data=tips, x='day', y='total_bill',
    hue='sex', dodge=True,
    order=['Thur', 'Fri', 'Sat', 'Sun'],
    markers=['o', 's'], linestyles=['-', '--'],
    palette='dark',
)
plt.title('Mean Total Bill by Day and Sex')
plt.show()

## Grouped Bar and Box Plots

Adding `hue` to `barplot()` or `boxplot()` produces side-by-side grouped comparisons.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar plot
sns.barplot(
    data=tips, x='day', y='tip',
    hue='time', order=['Thur', 'Fri', 'Sat', 'Sun'],
    palette='Set2', ax=axes[0],
)
axes[0].set_title('Mean Tip by Day and Meal Time')

# Grouped box plot
sns.boxplot(
    data=tips, x='day', y='tip',
    hue='time', order=['Thur', 'Fri', 'Sat', 'Sun'],
    palette='Set2', ax=axes[1],
)
axes[1].set_title('Tip Distribution by Day and Meal Time')

plt.tight_layout()
plt.show()

## Facet Grids via `relplot()` and `catplot()`

The `col` and `row` parameters in figure-level functions create small-multiple grids (facets). This is one of Seaborn's most powerful features for structured comparisons.

In [ ]:
# Line plot faceted by season using relplot
g = sns.relplot(
    data=flights, x='year', y='passengers',
    col='season', kind='line',
    col_wrap=2, height=3.5, aspect=1.3,
    col_order=['Winter', 'Spring', 'Summer', 'Fall'],
)
g.fig.suptitle('Passenger Trends by Season', y=1.03)
plt.show()

In [ ]:
# Categorical faceting with catplot
g = sns.catplot(
    data=tips, x='sex', y='total_bill',
    hue='smoker', col='time', row='day',
    kind='bar', height=3, aspect=1.2,
    palette='muted',
)
g.fig.suptitle('Average Bill: Sex x Smoker, Faceted by Time & Day', y=1.02)
plt.show()

In [ ]:
# Point plot via catplot for interaction visualization
g = sns.catplot(
    data=tips, x='day', y='tip',
    hue='sex', col='smoker',
    kind='point', dodge=True,
    order=['Thur', 'Fri', 'Sat', 'Sun'],
    height=4, aspect=1.2,
    palette='dark',
)
g.fig.suptitle('Mean Tip: Day x Sex, Smoker vs Non-Smoker', y=1.03)
plt.show()

## Practical Example: Flights Year-over-Year Comparison

In [ ]:
# Compare first 3 years vs last 3 years
flights['era'] = flights['year'].apply(
    lambda y: 'Early (1949-1951)' if y <= 1951 else (
        'Late (1958-1960)' if y >= 1958 else 'Middle'
    )
)

subset = flights[flights['era'] != 'Middle'].copy()

plt.figure(figsize=(10, 5))
sns.pointplot(
    data=subset, x='month', y='passengers',
    hue='era', dodge=True,
    markers=['o', 's'], linestyles=['-', '--'],
    palette='dark',
)
plt.title('Monthly Passengers: Early vs Late Period')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Faceted line comparison
g = sns.relplot(
    data=flights[flights['year'].isin([1949, 1954, 1960])],
    x='month', y='passengers',
    col='year', kind='line',
    marker='o', height=4, aspect=1.1,
)
for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=45)
g.fig.suptitle('Seasonal Pattern in Three Selected Years', y=1.03)
plt.show()

## Key Takeaways

| Function | Best For |
|---|---|
| `lineplot()` | Time series, continuous trends, CI bands |
| `pointplot()` | Comparing means across categories with connecting lines |
| `barplot(hue=...)` | Grouped bar comparison of means |
| `boxplot(hue=...)` | Grouped distribution comparison |
| `relplot(col=..., row=...)` | Faceted scatter or line plots |
| `catplot(col=..., row=...)` | Faceted categorical plots of any kind |

- `lineplot()` automatically computes confidence intervals when multiple observations share the same x-value.
- `pointplot()` is excellent for spotting **interaction effects** — non-parallel lines indicate that the effect of one variable depends on another.
- Faceting with `col` and `row` is the cleanest way to present multi-dimensional comparisons without overloading a single plot.